# Graph Analytics — Neo4j Graph Data Science

Demonstrate **graph algorithms** on the MovieLens + TMDB knowledge graph for **movie recommendation** use cases.

**Algorithms**
1. Degree Centrality
2. PageRank
3. Node Similarity
4. Louvain Community Detection

For each algorithm: theory → Cypher (GDS) → visualization → interpretation.

**Prerequisites**
- Graph loaded (`graph/build_graph.py`) and preferably TMDB-enriched (`graph/enrich_tmdb.py`)
- Neo4j with the **Graph Data Science** plugin (Community GDS is enough)
  - Local Docker: `NEO4J_PLUGINS` includes `graph-data-science` (see `docker-compose.yml`)
  - **Aura Free does not ship GDS** — use local Neo4j/Docker or AuraDS
- Credentials in `.env` (`NEO4J_URI`, `NEO4J_USERNAME` / `NEO4J_USER`, `NEO4J_PASSWORD`)

**Stack:** Neo4j GDS · Cypher · Pandas · Plotly

---
## 0. Setup & connectivity

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "graph"))
load_dotenv(ROOT / ".env")

from neo4j_config import connect_neo4j  # noqa: E402

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:.4f}".format)

PLOTLY_TEMPLATE = "plotly_white"
COLOR_PRIMARY = "#2E86AB"
COLOR_SECONDARY = "#E94F37"
COLOR_ACCENT = "#F6AE2D"
COLOR_SCALE = "Tealgrn"

TOP_N = 20
SEED_MOVIE_SUBSTRING = "Toy Story"  # used for similarity / community drill-downs

In [2]:
driver, DATABASE = connect_neo4j()


def run_cypher(query: str, parameters: dict | None = None) -> pd.DataFrame:
    """Execute Cypher and return a DataFrame."""
    with driver.session(database=DATABASE) as session:
        result = session.run(query, parameters or {})
        rows = [dict(record) for record in result]
    return pd.DataFrame(rows)


def run_void(query: str, parameters: dict | None = None) -> None:
    with driver.session(database=DATABASE) as session:
        session.run(query, parameters or {}).consume()


def drop_graph_if_exists(name: str) -> None:
    exists = run_cypher(
        "CALL gds.graph.exists($name) YIELD exists RETURN exists",
        {"name": name},
    )["exists"].iloc[0]
    if exists:
        run_void("CALL gds.graph.drop($name) YIELD graphName", {"name": name})
        print(f"Dropped existing projection: {name}")


overview = run_cypher(
    """
    MATCH (n)
    UNWIND labels(n) AS label
    RETURN label, count(*) AS n
    ORDER BY n DESC
    """
)
print(f"Connected · database={DATABASE!r}")
overview

Connected · database='neo4j'


,label,n
0,Actor,18480
1,Keyword,17326
2,Movie,9742
3,Director,4393
4,User,610
5,Genre,19


In [3]:
# Require Graph Data Science (fail early with actionable guidance)
try:
    gds = run_cypher("RETURN gds.version() AS gdsVersion")
    print(f"Neo4j GDS version: {gds['gdsVersion'].iloc[0]}")
except Exception as exc:
    raise SystemExit(
        "Neo4j Graph Data Science is not available on this instance.\n"
        "• Aura Free: GDS is not included — point .env at local Docker Neo4j, or use AuraDS.\n"
        "• Docker: ensure NEO4J_PLUGINS includes graph-data-science, then recreate the container "
        "(docker compose up -d --force-recreate neo4j).\n"
        f"Underlying error: {exc}"
    ) from exc


Neo4j GDS version: 2.13.11


### Graph projections used in this notebook

GDS algorithms run on **in-memory projections**, not directly on the store graph.

| Projection | Nodes | Relationships | Purpose |
|---|---|---|---|
| `rec_bipartite` | `User`, `Movie` | `RATED` (undirected) | popularity / influence from ratings |
| `rec_content` | `Movie`, `Genre`, `Actor`, `Director`, `Keyword` | content edges (undirected) | similarity & communities |

We drop-and-recreate projections so the notebook is idempotent.

In [4]:
drop_graph_if_exists("rec_bipartite")
drop_graph_if_exists("rec_content")

run_cypher(
    """
    CALL gds.graph.project(
      'rec_bipartite',
      ['User', 'Movie'],
      {
        RATED: {
          orientation: 'UNDIRECTED',
          properties: {
            weight: {
              property: 'rating',
              defaultValue: 3.0
            }
          }
        }
      }
    )
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount
    """
)

,graphName,nodeCount,relationshipCount
0,rec_bipartite,10352,201672


In [5]:
# Content graph may be empty if TMDB enrichment was skipped — check coverage first
enrichment = run_cypher(
    """
    MATCH (m:Movie)
    RETURN
      count(m) AS movies,
      sum(CASE WHEN m.tmdbEnriched THEN 1 ELSE 0 END) AS enriched
    """
)
print(enrichment.to_string(index=False))

content_edges = run_cypher(
    """
    MATCH ()-[r:HAS_GENRE|ACTED_BY|DIRECTED_BY|HAS_KEYWORD]->()
    RETURN count(r) AS n
    """
)["n"].iloc[0]

if content_edges == 0:
    raise SystemExit(
        "No content relationships found. Run: uv run python graph/enrich_tmdb.py"
    )

run_cypher(
    """
    CALL gds.graph.project(
      'rec_content',
      ['Movie', 'Genre', 'Actor', 'Director', 'Keyword'],
      {
        HAS_GENRE:   {orientation: 'UNDIRECTED'},
        ACTED_BY:    {orientation: 'UNDIRECTED'},
        DIRECTED_BY: {orientation: 'UNDIRECTED'},
        HAS_KEYWORD: {orientation: 'UNDIRECTED'}
      }
    )
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount
    """
)

 movies  enriched
   9742      9621


,graphName,nodeCount,relationshipCount
0,rec_content,49960,373116


---
## 1. Degree Centrality

### Theory

**Degree centrality** scores a node by how many edges it has (optionally weighted).

In an undirected bipartite rating graph `(User)—[:RATED]—(Movie)`:

- **Movie degree** ≈ how many users rated it → a direct **popularity** signal  
- **User degree** ≈ how many movies that user rated → activity / cold-start risk  

Weighted degree (sum of `rating`) favors titles that not only are widely rated but also rated highly on average volume.

**Recommendation use cases**
- Non-personalized “trending / most watched” baselines  
- Popularity prior for hybrid rankers  
- Detect hubs that dominate collaborative paths (and may need diversification)

In [6]:
degree_movies = run_cypher(
    """
    CALL gds.degree.stream('rec_bipartite')
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WHERE 'Movie' IN labels(n)
    RETURN n.movieId AS movieId, n.title AS title, score AS degree
    ORDER BY degree DESC
    LIMIT $topN
    """,
    {"topN": TOP_N},
)

degree_users = run_cypher(
    """
    CALL gds.degree.stream('rec_bipartite')
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WHERE 'User' IN labels(n)
    RETURN n.userId AS userId, score AS degree
    ORDER BY degree DESC
    LIMIT $topN
    """,
    {"topN": TOP_N},
)

degree_weighted = run_cypher(
    """
    CALL gds.degree.stream('rec_bipartite', {relationshipWeightProperty: 'weight'})
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WHERE 'Movie' IN labels(n)
    RETURN n.movieId AS movieId, n.title AS title, score AS weightedDegree
    ORDER BY weightedDegree DESC
    LIMIT $topN
    """,
    {"topN": TOP_N},
)

print("Top movies by degree (rating count)")
display(degree_movies.head(10))
print("Top users by activity")
display(degree_users.head(10))

Top movies by degree (rating count)


,movieId,title,degree
0,356,Forrest Gump (1994),329.0000
1,318,"Shawshank Redemption, The (1994)",317.0000
2,296,Pulp Fiction (1994),307.0000
3,593,"Silence of the Lambs, The (1991)",279.0000
4,2571,"Matrix, The (1999)",278.0000
5,260,Star Wars: Episode IV - A New Hope (1977),251.0000
6,480,Jurassic Park (1993),238.0000
7,110,Braveheart (1995),237.0000
8,589,Terminator 2: Judgment Day (1991),224.0000
9,527,Schindler's List (1993),220.0000


Top users by activity


,userId,degree
0,414,2698.0000
1,599,2478.0000
2,474,2108.0000
3,448,1864.0000
4,274,1346.0000
5,610,1302.0000
6,68,1260.0000
7,380,1218.0000
8,606,1115.0000
9,288,1055.0000


In [7]:
fig = px.bar(
    degree_movies.sort_values("degree"),
    x="degree",
    y="title",
    orientation="h",
    title="Degree Centrality — most-rated movies",
    labels={"degree": "Degree (n ratings)", "title": "Movie"},
    color="degree",
    color_continuous_scale=COLOR_SCALE,
    template=PLOTLY_TEMPLATE,
    height=560,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False)
fig.show()

fig2 = px.bar(
    degree_weighted.sort_values("weightedDegree"),
    x="weightedDegree",
    y="title",
    orientation="h",
    title="Weighted Degree — sum of ratings",
    labels={"weightedDegree": "Σ ratings", "title": "Movie"},
    color_discrete_sequence=[COLOR_SECONDARY],
    template=PLOTLY_TEMPLATE,
    height=560,
)
fig2.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig2.show()

### Interpretation — Degree Centrality

- The head of the degree ranking is the **catalog popularity backbone**: titles that appear in almost every collaborative neighborhood.
- High-degree movies are excellent **cold-start fillers** and strong features for “people also watched”, but they hurt **novelty** if over-weighted.
- Power-user tails (very high user degree) can dominate co-rating similarity — consider damping or min/max rating filters in production CF.
- Weighted degree shifts ranking toward titles with **many strong ratings**, not only many ratings — closer to a quality-aware popularity prior.

---
## 2. PageRank

### Theory

**PageRank** estimates importance via a random walk with restart: a node is important if important nodes point to it (or, on undirected graphs, if it sits on many high-traffic paths).

On `rec_bipartite`, PageRank mixes user activity and movie popularity into a single **influence** score. On `rec_content`, it surfaces **central cultural entities** (genres, actors, directors) and movies that sit at content hubs.

**Recommendation use cases**
- Soft ranking feature beyond raw popularity  
- Identify influential cast/crew for explanation and editorial rails  
- Prioritize graph expansion / enrichment for high-influence nodes

In [8]:
pagerank_movies = run_cypher(
    """
    CALL gds.pageRank.stream('rec_bipartite', {
      maxIterations: 20,
      dampingFactor: 0.85,
      relationshipWeightProperty: 'weight'
    })
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WHERE 'Movie' IN labels(n)
    RETURN n.movieId AS movieId, n.title AS title, score AS pageRank
    ORDER BY pageRank DESC
    LIMIT $topN
    """,
    {"topN": TOP_N},
)

pagerank_content = run_cypher(
    """
    CALL gds.pageRank.stream('rec_content', {
      maxIterations: 20,
      dampingFactor: 0.85
    })
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WITH n, score, labels(n) AS labs
    RETURN
      labs[0] AS label,
      coalesce(n.title, n.name) AS name,
      score AS pageRank
    ORDER BY pageRank DESC
    LIMIT $topN
    """,
    {"topN": TOP_N},
)

print("PageRank on rating graph (movies)")
display(pagerank_movies.head(10))
print("PageRank on content graph (any label)")
display(pagerank_content.head(10))

PageRank on rating graph (movies)


,movieId,title,pageRank
0,318,"Shawshank Redemption, The (1994)",13.2543
1,356,Forrest Gump (1994),13.0670
2,296,Pulp Fiction (1994),12.3961
3,2571,"Matrix, The (1999)",11.4841
4,593,"Silence of the Lambs, The (1991)",11.1901
5,260,Star Wars: Episode IV - A New Hope (1977),10.4957
6,2959,Fight Club (1999),9.3105
7,527,Schindler's List (1993),8.9203
8,110,Braveheart (1995),8.8844
9,1196,Star Wars: Episode V - The Empire Strikes Back (1980),8.8187


PageRank on content graph (any label)


,label,name,pageRank
0,Genre,Drama,473.9174
1,Genre,Comedy,425.7746
2,Genre,Romance,202.0593
3,Genre,Thriller,194.1307
4,Genre,Action,182.8033
5,Genre,Crime,144.8628
6,Genre,Adventure,133.5457
7,Genre,Horror,111.7244
8,Genre,Science Fiction,108.3530
9,Keyword,based on novel or book,102.3511


In [9]:
compare = degree_movies[["movieId", "title", "degree"]].merge(
    pagerank_movies[["movieId", "pageRank"]],
    on="movieId",
    how="outer",
).dropna()

fig = px.scatter(
    compare,
    x="degree",
    y="pageRank",
    text="title",
    title="Degree vs PageRank (rating graph)",
    labels={"degree": "Degree", "pageRank": "PageRank"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
    height=520,
)
fig.update_traces(textposition="top center", textfont_size=9)
fig.show()

fig_c = px.bar(
    pagerank_content.sort_values("pageRank"),
    x="pageRank",
    y="name",
    color="label",
    orientation="h",
    title="PageRank hubs on the content graph",
    template=PLOTLY_TEMPLATE,
    height=560,
)
fig_c.update_layout(yaxis={"categoryorder": "total ascending"})
fig_c.show()

### Interpretation — PageRank

- When Degree and PageRank **agree**, the catalog has a clear popularity core.
- Movies with **high PageRank relative to degree** sit next to influential users/neighbors — useful “connector” titles for exploration.
- On the content graph, top PageRank nodes are often **broad genres or prolific actors** — good for explanations (“because you like Drama / this cast”), weaker as the sole ranking signal.
- Use PageRank as a **feature**, not the only score: combine with taste similarity to avoid always recommending the same hubs.

---
## 3. Node Similarity

### Theory

**Node Similarity** (Jaccard by default in GDS) compares nodes by shared neighbors:

\[
J(u,v) = \frac{|N(u) \cap N(v)|}{|N(u) \cup N(v)|}
\]

On `rec_content`, two movies are similar if they share genres, cast, directors, or keywords.  
On `rec_bipartite`, two movies are similar if they share raters (collaborative signal).

**Recommendation use cases**
- Item–item “more like this”  
- Seed expansion for GraphRAG / path explanations (`docs/graph_rag.md`)
- Content-based fallback when collaborative data is sparse

In [10]:
# Content-based movie–movie similarity
sim_content = run_cypher(
    """
    CALL gds.nodeSimilarity.stream('rec_content', {
      topK: 10,
      similarityCutoff: 0.1
    })
    YIELD node1, node2, similarity
    WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
    WHERE 'Movie' IN labels(a) AND 'Movie' IN labels(b)
    RETURN
      a.movieId AS movieId1,
      a.title AS title1,
      b.movieId AS movieId2,
      b.title AS title2,
      similarity
    ORDER BY similarity DESC
    LIMIT 40
    """
)

print(f"Content similarity pairs: {len(sim_content)}")
sim_content.head(15)

Content similarity pairs: 40


,movieId1,title1,movieId2,title2,similarity
0,78544,Ricky Gervais Live 3: Fame (2007),78160,Ricky Gervais Live: Animals (2003),1.0000
1,121372,Bill Burr: Let It Go (2010),121374,Bill Burr: Why Do I Do This? (2008),1.0000
2,144606,Confessions of a Dangerous Mind (2002),6003,Confessions of a Dangerous Mind (2002),1.0000
3,6003,Confessions of a Dangerous Mind (2002),144606,Confessions of a Dangerous Mind (2002),1.0000
4,147326,The Adventures of Sherlock Holmes and Doctor Watson: King of Blackmailers (1980),147300,Adventures Of Sherlock Holmes And Dr. Watson: The Twentieth Century Approaches (1986),1.0000
5,151769,Three from Prostokvashino (1978),172587,Vacations in Prostokvashino (1980),1.0000
6,168632,Bill Burr: Walk Your Way Out (2017),119153,Bill Burr: You People Are All the Same (2012),1.0000
7,172589,Winter in Prostokvashino (1984),151769,Three from Prostokvashino (1978),1.0000
8,183227,Dave Chappelle: The Bird Revelation (2017),170411,Dave Chappelle: Deep in the Heart of Texas (2017),1.0000
9,172587,Vacations in Prostokvashino (1980),172589,Winter in Prostokvashino (1984),1.0000


In [11]:
# Collaborative movie–movie similarity (shared raters)
sim_collab = run_cypher(
    """
    CALL gds.nodeSimilarity.stream('rec_bipartite', {
      topK: 8,
      similarityCutoff: 0.05
    })
    YIELD node1, node2, similarity
    WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
    WHERE 'Movie' IN labels(a) AND 'Movie' IN labels(b)
    RETURN
      a.movieId AS movieId1,
      a.title AS title1,
      b.movieId AS movieId2,
      b.title AS title2,
      similarity
    ORDER BY similarity DESC
    LIMIT 40
    """
)

print(f"Collaborative similarity pairs: {len(sim_collab)}")
sim_collab.head(15)

Collaborative similarity pairs: 40


,movieId1,title1,movieId2,title2,similarity
0,108,Catwalk (1996),602,"Great Day in Harlem, A (1994)",1.0000
1,117,"Young Poisoner's Handbook, The (1995)",285,Beyond Bedlam (1993),1.0000
2,108,Catwalk (1996),320,Suture (1993),1.0000
3,108,Catwalk (1996),649,Cold Fever (Á köldum klaka) (1995),1.0000
4,108,Catwalk (1996),526,"Savage Nights (Nuits fauves, Les) (1992)",1.0000
5,108,Catwalk (1996),77,Nico Icon (1995),1.0000
6,77,Nico Icon (1995),149,Amateur (1994),1.0000
7,77,Nico Icon (1995),526,"Savage Nights (Nuits fauves, Les) (1992)",1.0000
8,96,In the Bleak Midwinter (1995),982,Picnic (1955),1.0000
9,108,Catwalk (1996),301,Picture Bride (Bijo photo) (1994),1.0000


In [12]:
# Drill-down: neighbors of a seed title (content similarity)
seed_rows = run_cypher(
    """
    MATCH (m:Movie)
    WHERE toLower(m.title) CONTAINS toLower($q)
    RETURN m.movieId AS movieId, m.title AS title
    ORDER BY size(m.title) ASC
    LIMIT 5
    """,
    {"q": SEED_MOVIE_SUBSTRING},
)
display(seed_rows)

if seed_rows.empty:
    print("Seed movie not found — change SEED_MOVIE_SUBSTRING")
else:
    seed_id = int(seed_rows.iloc[0]["movieId"])
    seed_title = seed_rows.iloc[0]["title"]

    seed_sims = run_cypher(
        """
        CALL gds.nodeSimilarity.stream('rec_content', {
          topK: 15,
          similarityCutoff: 0.05
        })
        YIELD node1, node2, similarity
        WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
        WHERE 'Movie' IN labels(a) AND 'Movie' IN labels(b)
          AND (a.movieId = $seedId OR b.movieId = $seedId)
        RETURN
          CASE WHEN a.movieId = $seedId THEN b.movieId ELSE a.movieId END AS movieId,
          CASE WHEN a.movieId = $seedId THEN b.title ELSE a.title END AS title,
          similarity
        ORDER BY similarity DESC
        LIMIT 15
        """,
        {"seedId": seed_id},
    )

    fig = px.bar(
        seed_sims.sort_values("similarity"),
        x="similarity",
        y="title",
        orientation="h",
        title=f"Content Node Similarity — neighbors of {seed_title}",
        color="similarity",
        color_continuous_scale=COLOR_SCALE,
        template=PLOTLY_TEMPLATE,
        height=520,
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False)
    fig.show()
    display(seed_sims)

,movieId,title
0,1,Toy Story (1995)
1,3114,Toy Story 2 (1999)
2,78499,Toy Story 3 (2010)


[#CA40]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0))): OSError('No data')


ServiceUnavailable: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0)))

In [ ]:
# Shared-neighbor evidence for the top content pair (explainability)
if not sim_content.empty:
    t1, t2 = sim_content.iloc[0]["title1"], sim_content.iloc[0]["title2"]
    evidence = run_cypher(
        """
        MATCH (a:Movie {title: $t1})-[r1]->(x)<-[r2]-(b:Movie {title: $t2})
        WHERE type(r1) IN ['HAS_GENRE','ACTED_BY','DIRECTED_BY','HAS_KEYWORD']
          AND type(r1) = type(r2)
        RETURN type(r1) AS rel, coalesce(x.name, x.title) AS shared, labels(x)[0] AS kind
        ORDER BY rel, shared
        LIMIT 30
        """,
        {"t1": t1, "t2": t2},
    )
    print(f"Why similar? {t1} ↔ {t2}")
    display(evidence)

Why similar? Ricky Gervais Live 3: Fame (2007) ↔ Ricky Gervais Live: Animals (2003)


,rel,shared,kind
0,ACTED_BY,Ricky Gervais,Actor
1,DIRECTED_BY,Dominic Brigstocke,Director
2,HAS_GENRE,Comedy,Genre
3,HAS_KEYWORD,stand-up comedy,Keyword


### Interpretation — Node Similarity

- **Content similarity** recovers franchise / genre / cast clusters — ideal for “more like this” when metadata is rich.
- **Collaborative similarity** recovers audience taste overlap — can link films that look different on paper but share fans.
- Disagreement between the two signals is a feature: hybrid recommenders can blend them or show both explanation types.
- Always inspect shared neighbors (cell above) — Jaccard without evidence is hard to trust in a product UI.

---
## 4. Louvain Community Detection

### Theory

**Louvain** greedily maximizes **modularity**: partitions the graph so that intra-community edges are denser than a random null model.

On `rec_content`, communities are clusters of movies (and bridging entities) that share dense metadata neighborhoods — soft “taste worlds” or franchise/genre ecosystems.

**Recommendation use cases**
- Diversify recommendations across communities  
- Serendipity: suggest another movie from the user’s preferred cluster  
- Editorial shelves / facet navigation without hand-built taxonomies  
- Detect near-duplicate clusters for catalog QA

In [ ]:
# Write community ids onto nodes for easier joins / Browser viz
run_void(
    """
    CALL gds.louvain.write('rec_content', {
      writeProperty: 'louvainCommunity',
      maxLevels: 10,
      tolerance: 0.0001
    })
    """
)

communities = run_cypher(
    """
    MATCH (m:Movie)
    WHERE m.louvainCommunity IS NOT NULL
    RETURN
      m.louvainCommunity AS community,
      count(*) AS nMovies
    ORDER BY nMovies DESC
    """
)

print(f"Communities with ≥1 movie: {len(communities)}")
print(f"Largest community size: {int(communities['nMovies'].iloc[0]) if not communities.empty else 0}")
communities.head(15)

Communities with ≥1 movie: 198
Largest community size: 1837


,community,nMovies
0,42326,1837
1,3456,1821
2,2775,1132
3,40560,896
4,37410,887
5,6113,647
6,44555,383
7,25298,365
8,45456,203
9,4157,128


In [ ]:
# Characterize top communities by dominant genres
community_genres = run_cypher(
    """
    MATCH (m:Movie)-[:HAS_GENRE]->(g:Genre)
    WHERE m.louvainCommunity IS NOT NULL
    WITH m.louvainCommunity AS community, g.name AS genre, count(*) AS n
    ORDER BY community, n DESC
    WITH community, collect({genre: genre, n: n})[0..5] AS topGenres, sum(n) AS total
    RETURN community, topGenres, total
    ORDER BY total DESC
    LIMIT 12
    """
)

rows = []
for _, row in community_genres.iterrows():
    top = row["topGenres"] or []
    label = ", ".join(f"{t['genre']} ({t['n']})" for t in top[:3])
    rows.append(
        {
            "community": int(row["community"]),
            "moviesLinks": int(row["total"]),
            "topGenres": label,
        }
    )
community_summary = pd.DataFrame(rows)
display(community_summary)

fig = px.bar(
    communities.head(15),
    x="community",
    y="nMovies",
    title="Louvain — movie count per community (top 15)",
    labels={"nMovies": "# movies", "community": "Community id"},
    color="nMovies",
    color_continuous_scale=COLOR_SCALE,
    template=PLOTLY_TEMPLATE,
    height=420,
)
fig.update_layout(coloraxis_showscale=False, xaxis_type="category")
fig.show()

,community,moviesLinks,topGenres
0,3456,4974,"Thriller (1060), Drama (968), Crime (964)"
1,42326,4212,"Comedy (1428), Drama (965), Romance (943)"
2,40560,2877,"Family (566), Adventure (466), Comedy (433)"
3,2775,2682,"Drama (1000), History (318), War (278)"
4,37410,2567,"Science Fiction (635), Action (329), Horror (325)"
5,6113,1828,"Action (409), Comedy (224), Adventure (190)"
6,44555,974,"Drama (155), Comedy (150), Thriller (132)"
7,25298,620,"Documentary (251), Comedy (86), Music (80)"
8,45456,490,"Drama (114), Comedy (77), Action (56)"
9,18958,303,"Western (63), Drama (58), Comedy (42)"


In [ ]:
# Sample titles from the largest communities
samples = run_cypher(
    """
    MATCH (m:Movie)
    WHERE m.louvainCommunity IS NOT NULL
    WITH m.louvainCommunity AS community, collect(m.title)[0..8] AS sampleTitles, count(*) AS n
    ORDER BY n DESC
    RETURN community, n AS nMovies, sampleTitles
    LIMIT 8
    """
)
samples

,community,nMovies,sampleTitles
0,42326,1837,"[Toy Story (1995), Grumpier Old Men (1995), Waiting to Exhale (1995), Father of the Bride Part I..."
1,3456,1821,"[Heat (1995), Sudden Death (1995), Nixon (1995), Casino (1995), Four Rooms (1995), Money Train (..."
2,2775,1132,"[Sense and Sensibility (1995), Othello (1995), Shanghai Triad (Yao a yao yao dao waipo qiao) (19..."
3,40560,896,"[Jumanji (1995), Tom and Huck (1995), Balto (1995), Ace Ventura: When Nature Calls (1995), Babe ..."
4,37410,887,"[Powder (1995), City of Lost Children, The (Cité des enfants perdus, La) (1995), Bio-Dome (1996)..."
5,6113,647,"[Mortal Kombat (1995), Batman Forever (1995), Casper (1995), Lord of Illusions (1995), Mighty Mo..."
6,44555,383,"[GoldenEye (1995), Cutthroat Island (1995), Awfully Big Adventure, An (1995), First Knight (1995..."
7,25298,365,"[Heidi Fleiss: Hollywood Madam (1995), Catwalk (1996), Jupiter's Wife (1994), Crumb (1994), Unzi..."


In [ ]:
# Which community does the seed movie belong to? Suggest peers from the same cluster.
if not seed_rows.empty:
    seed_id = int(seed_rows.iloc[0]["movieId"])
    peers = run_cypher(
        """
        MATCH (seed:Movie {movieId: $seedId})
        WHERE seed.louvainCommunity IS NOT NULL
        MATCH (other:Movie {louvainCommunity: seed.louvainCommunity})
        WHERE other.movieId <> seed.movieId
        OPTIONAL MATCH (:User)-[r:RATED]->(other)
        WITH seed, other, count(r) AS nRatings, avg(r.rating) AS avgRating
        RETURN
          seed.title AS seed,
          seed.louvainCommunity AS community,
          other.title AS peer,
          nRatings,
          round(avgRating, 3) AS avgRating
        ORDER BY nRatings DESC
        LIMIT 15
        """,
        {"seedId": seed_id},
    )
    display(peers)

    if not peers.empty:
        fig = px.bar(
            peers.sort_values("nRatings"),
            x="nRatings",
            y="peer",
            orientation="h",
            title=f"Community peers of {peers.iloc[0]['seed']} (id={int(peers.iloc[0]['community'])})",
            color="avgRating",
            color_continuous_scale="Tealgrn",
            template=PLOTLY_TEMPLATE,
            height=520,
        )
        fig.update_layout(yaxis={"categoryorder": "total ascending"})
        fig.show()

,seed,community,peer,nRatings,avgRating
0,Toy Story (1995),42326,Forrest Gump (1994),329,4.1640
1,Toy Story (1995),42326,American Beauty (1999),204,4.0560
2,Toy Story (1995),42326,Pirates of the Caribbean: The Curse of the Black Pearl (2003),149,3.7790
3,Toy Story (1995),42326,Mrs. Doubtfire (1993),144,3.3890
4,Toy Story (1995),42326,Groundhog Day (1993),143,3.9440
5,Toy Story (1995),42326,Pretty Woman (1990),135,3.4850
6,Toy Story (1995),42326,One Flew Over the Cuckoo's Nest (1975),133,4.2030
7,Toy Story (1995),42326,Dumb & Dumber (Dumb and Dumber) (1994),133,3.0600
8,Toy Story (1995),42326,Eternal Sunshine of the Spotless Mind (2004),131,4.1600
9,Toy Story (1995),42326,"Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)",120,4.1830


### Interpretation — Louvain

- Large communities usually align with **broad genres or franchises**; tiny communities are niche metadata islands (useful for long-tail discovery).
- Recommending **within** a user’s dominant communities improves relevance; forcing **cross-community** picks increases diversity / serendipity.
- Community id alone is opaque — always attach genre/cast summaries (as above) before shipping to a UI.
- Writing `louvainCommunity` back to Neo4j lets Streamlit / GraphRAG reuse clusters without re-running GDS every request.

---
## 5. Putting it together — recommendation playbook

| Signal | Algorithm | Product role |
|---|---|---|
| Popularity prior | Degree | Cold-start, trending rails |
| Global influence | PageRank | Soft rank feature / hub detection |
| “More like this” | Node Similarity | Item–item CF / content neighbors |
| Taste worlds | Louvain | Diversification & cluster shelves |

**Practical hybrid sketch**

1. Candidate generation: Node Similarity (content + collab) ∪ Louvain peers  
2. Re-rank: personalized score × PageRank<sup>α</sup> × Degree<sup>β</sup> with small α, β  
3. Diversify: MMR or “at most *k* from the same Louvain community”  
4. Explain: shared neighbors (similarity) + community genre summary

The shipping ranker in this repo is the multi-signal path scorer (`docs/graph_recommender.md`), not a GDS re-ranker. Conversational QA is GraphRAG (`docs/graph_rag.md`) — better than chunk RAG when the answer is a relationship.

In [ ]:
# Optional cleanup of in-memory projections (node properties like louvainCommunity remain)
drop_graph_if_exists("rec_bipartite")
drop_graph_if_exists("rec_content")
driver.close()
print("Projections dropped · driver closed.")
print("Note: Movie.louvainCommunity properties were written to the store — drop manually if undesired:")
print("  MATCH (n) WHERE n.louvainCommunity IS NOT NULL REMOVE n.louvainCommunity")

Dropped existing projection: rec_bipartite
Dropped existing projection: rec_content
Projections dropped · driver closed.
Note: Movie.louvainCommunity properties were written to the store — drop manually if undesired:
  MATCH (n) WHERE n.louvainCommunity IS NOT NULL REMOVE n.louvainCommunity


---
## 6. Insights

1. **Degree** recovers the MovieLens popularity head — necessary baseline, dangerous if used alone.  
2. **PageRank** highlights structural hubs; correlate with degree to find “bridge” titles.  
3. **Node Similarity** on content edges yields explainable neighbors; collaborative similarity captures taste, not plot.  
4. **Louvain** soft-clusters the catalog into reusable communities for diversification and shelves.  
5. GDS projections keep analytics **off the hot transactional path** — project → compute → write selected properties → drop graph.

**Next steps in this repo:** path-ranker holdout (`docs/graph_recommender.md`) and GraphRAG vs vector RAG (`docs/graph_rag.md`), both in `notebooks/05_evaluation.ipynb`. Optional GDS follow-ups: persist `SIMILAR_TO`, expose community filters in Streamlit.

Running this notebook writes `Movie.louvainCommunity` in Neo4j. That does **not** change `eval/latest_results.json`.